## Aquecendo


In [0]:
SELECT current_catalog(), current_schema()


current_catalog(),current_schema()
workspace,default


In [0]:
USE CATALOG smart_claims_dev

In [0]:
SELECT current_catalog(), current_schema()


current_catalog(),current_schema()
smart_claims_dev,default


In [0]:
USE SCHEMA 01_bronze

In [0]:
DROP TABLE IF EXISTS claims

## EXPLORE THE DATA SOURCE FILES



Query os arquivos CSV usando o path do volume

In [0]:
SELECT *
FROM csv.`/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv`
--drop the table if it exists for demonstration

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_c16,_c17,_c18
claim_no,policy_no,claim_date,months_as_customer,injury,property,vehicle,total,collision_type,number_of_vehicles_involved,age,insured_relationship,license_issue_date,date,hour,type,severity,number_of_witnesses,suspicious_activity
f59f665f-c82f-4aa3-98b5-060c7a8f6d19,102049729,2016-01-26,259,780,780,3510,5070,Side Collision,4,40.0,unmarried,24-05-2006,2016-10-14,10,Multi-vehicle Collision,Minor Damage,0,false
f051f2ae-c92e-44f9-9bb8-3ab60b19341b,102147750,2020-08-18,261,9680,14520,38720,62920,Side Collision,4,24.0,other-relative,25-10-2016,2021-05-09,7,Multi-vehicle Collision,Major Damage,2,false
1aec2184-3a54-4b3b-a41b-51b79feb4538,102139065,2020-01-20,86,4060,4060,24360,32480,Rear Collision,1,32.0,own-child,22-12-2008,2020-10-11,3,Parked Car,Total Loss,0,false
22b8e795-10e8-4981-b62c-59607a215655,102099757,2017-04-12,121,670,670,4690,6030,Rear Collision,1,23.0,other-relative,12-08-2012,2017-11-27,21,Parked Car,Total Loss,0,false
5ccc8c8f-d199-4cad-af49-90e45a729489,102071474,2017-01-22,215,15620,7810,54670,78100,Front Collision,1,26.0,husband,08-08-2012,2017-05-21,9,Single Vehicle Collision,Major Damage,3,true
7730a464-78e3-49ee-98b4-40072c13f492,102136098,2019-10-14,290,4570,4570,31990,41130,Front Collision,4,27.0,unmarried,26-01-2014,2020-03-30,19,Multi-vehicle Collision,Major Damage,2,false
6bbca89a-4c02-4beb-ab06-63d301f6f39e,102106062,2017-06-17,371,4730,4730,37840,47300,Rear Collision,1,30.0,not-in-family,07-01-2010,2018-02-03,2,Single Vehicle Collision,Major Damage,0,true
25144193-d014-44d7-ad7d-771337bd294c,102131860,2019-07-06,338,5190,10380,36330,51900,Side Collision,4,37.0,unmarried,01-05-2016,2020-02-22,21,Multi-vehicle Collision,Minor Damage,3,true
fae3651c-f0c8-421b-bbbe-3483d6c03def,102097981,2017-04-12,190,4940,9880,34580,49400,Front Collision,4,23.0,wife,29-11-2016,2017-12-03,22,Multi-vehicle Collision,Minor Damage,1,false


## Create a STREAMING TABLE using Databricks Auto Loader

In [0]:
CREATE OR REFRESH STREAMING TABLE smart_claims_dev.01_bronze.claims
SCHEDULE EVERY 1 WEEK
AS
SELECT *
FROM read_files(
  '/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv',
  format => 'csv',
  header => true
)



---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8108989708214317>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE OR REFRESH STREAMING TABLE smart_claims_dev.01_bronze.claims\nSCHEDULE EVERY 1 WEEK\nAS\nSELECT *\nFROM read_files(\n  '/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv',\n  format => 'csv',\n  header => true\n)\n\n")

File /databricks/python/lib/python3.11/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCE

In [0]:
CREATE OR REFRESH STREAMING TABLE smart_claims_dev.01_bronze.claims
AS
SELECT *
FROM STREAM cloud_files(
  '/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv',
  format => 'csv',
  header => true
)

In [0]:
-- This file defines a sample transformation.
-- Edit the sample below or add new transformations
-- using "+ Add" in the file browser.

CREATE OR REFRESH STREAMING LIVE TABLE smart_claims_dev.01_bronze.claims
COMMENT "Bronze — sinistros (claims) a partir de 00_landing/sql_server/claims.csv"
TBLPROPERTIES ("quality" = "bronze")
AS
SELECT
  *,
  _metadata.file_path               AS input_file,
  _metadata.file_modification_time  AS input_file_mtime,
  current_timestamp()               AS ingested_at
FROM cloud_files(
  "/Volumes/smart_claims_dev/00_landing/sql_server",
  "csv",
  map(
    "header", "true",
    "cloudFiles.inferColumnTypes", "true",
    "cloudFiles.schemaEvolutionMode", "addNewColumns",
    "pathGlobFilter", "claims.csv"
  )
);

In [0]:
SELECT * FROM smart_claims_dev.01_bronze.claims
LIMIT 10;

claim_no,policy_no,claim_date,months_as_customer,injury,property,vehicle,total,collision_type,number_of_vehicles_involved,age,insured_relationship,license_issue_date,date,hour,type,severity,number_of_witnesses,suspicious_activity,_rescued_data,input_file,input_file_mtime,ingested_at
366369e0-60c5-4658-a221-96f88472439f,102143388,2020-04-08,204,9600,4800,33600,48000,null,4,33.0,other-relative,2015-01-06,2020-04-07,21,Multi-vehicle Collision,Major Damage,1,false,null,/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv,2026-10-02T03:32:15.000Z,2026-10-10T05:29:06.031Z
6dbc1f41-57d3-4798-9bc4-32a4a08cd1b3,102095813,2017-05-15,153,960,960,3840,5760,null,4,33.0,wife,2015-01-06,2017-05-04,23,Multi-vehicle Collision,Minor Damage,3,true,null,/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv,2026-10-02T03:32:15.000Z,2026-10-10T05:29:06.031Z
54d1da92-560a-439c-a4f5-b8889d338cdb,102139457,2020-01-22,141,12500,12500,43750,68750,Front Collision,1,32.0,own-child,2013-08-24,2020-01-15,23,Single Vehicle Collision,Major Damage,2,false,null,/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv,2026-10-02T03:32:15.000Z,2026-10-10T05:29:06.031Z
26db9d60-c1d5-492a-9f31-bcb1533e1305,102085741,2017-05-15,242,16800,16800,67200,100800,Rear Collision,4,32.0,own-child,2016-05-17,2016-08-21,2,Multi-vehicle Collision,Major Damage,2,false,null,/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv,2026-10-02T03:32:15.000Z,2026-10-10T05:29:06.031Z
198bc288-53c7-4a66-9a9d-5be7d872dc49,102065376,2016-11-11,399,4030,8060,36270,48360,Rear Collision,4,41.0,other-relative,2016-05-30,2016-11-07,3,Multi-vehicle Collision,Total Loss,3,true,null,/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv,2026-10-02T03:32:15.000Z,2026-10-10T05:29:06.031Z
63ff7f90-4a15-4e6d-af59-eb4a1305cebd,102050761,2016-04-01,405,7370,7370,58960,73700,null,4,25.0,not-in-family,2014-03-10,2015-12-18,21,Multi-vehicle Collision,Total Loss,2,false,null,/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv,2026-10-02T03:32:15.000Z,2026-10-10T05:29:06.031Z
4f02ddb5-801e-44d7-b818-ee60f87da728,102132459,2019-07-19,316,6230,12460,37380,56070,null,1,58.0,wife,1989-01-22,2019-07-13,17,Parked Car,Major Damage,1,false,null,/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv,2026-10-02T03:32:15.000Z,2026-10-10T05:29:06.031Z
f4a42dd6-1c01-47ba-8dc4-6e6c295722a9,102113658,2018-01-03,47,7080,21240,35400,63720,null,4,21.0,other-relative,2015-06-28,2017-11-28,17,Multi-vehicle Collision,Total Loss,3,false,null,/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv,2026-10-02T03:32:15.000Z,2026-10-10T05:29:06.031Z
50feb617-02c0-42fa-9653-208b695d38d3,102098557,2017-05-16,163,7240,14480,50680,72400,Rear Collision,1,25.0,other-relative,2016-12-25,2017-05-10,18,Vehicle Theft,Major Damage,1,true,null,/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv,2026-10-02T03:32:15.000Z,2026-10-10T05:29:06.031Z
fb6794d9-10d8-41b3-b346-6cbbadd89e71,102072953,2017-02-07,31,8960,8960,53760,71680,Front Collision,4,34.0,own-child,2016-04-28,2017-02-02,18,Multi-vehicle Collision,Major Damage,3,false,null,/Volumes/smart_claims_dev/00_landing/sql_server/claims.csv,2026-10-02T03:32:15.000Z,2026-10-10T05:29:06.031Z


In [0]:
DESCRIBE TABLE EXTENDED smart_claims_dev.01_bronze.claims;

col_name,data_type,comment
claim_no,string,null
policy_no,string,null
claim_date,string,null
months_as_customer,string,null
injury,string,null
property,string,null
vehicle,string,null
total,string,null
collision_type,string,null
number_of_vehicles_involved,string,null


In [0]:
DESCRIBE HISTORY smart_claims_dev.01_bronze.claims;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-10T05:53:17.000Z,74778899707691,renatozfilho20@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 22b320c8-3a1b-4691-9f8c-018f2e319298, epochId -> 0, statsOnLoad -> true)",null,null,393e818b-91d1-40eb-9448-25fa350c1f9d,1010-051943-vkcpjabi-v2n,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 12991, numOutputBytes -> 528736, numAddedFiles -> 1)",null,Databricks-Runtime/dlt:18.3.7-delta-pipelines-aarch64-photon-dlt-release-dp-20260929-rc0-commit-e10efbb-image-5340b25
1,2026-10-10T05:53:10.001Z,74778899707691,renatozfilho20@gmail.com,DLT SETUP,"Map(pipelineId -> 4c79ab57-fdd3-43bf-912d-5fff2770d5f1, updateId -> 139fefd2-aa2f-4c42-9c88-2883b2cf7e1f)",null,null,null,1010-051943-vkcpjabi-v2n,0,WriteSerializable,false,Map(),null,Databricks-Runtime/dlt:18.3.7-delta-pipelines-aarch64-photon-dlt-release-dp-20260929-rc0-commit-e10efbb-image-5340b25
0,2026-10-10T05:53:10.000Z,74778899707691,renatozfilho20@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""spark.sql.internal.pipelines.parentTableId"":""201f9bb9-565d-4888-8e6f-b8ae95ba61d7"",""delta.enableDeletionVectors"":""true"",""spark.sql.internal.unityCatalog.internalEntity.isListable"":""false"",""spark.sql.internal.unityCatalog.internalEntity.inheritsPolicy"":""false"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""b98b8e5e-ed63-40a1-8141-b11a9960eade"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-fb2559a5-97aa-4c70-9c9a-73a455930263"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-28fc9c64-95be-4c28-93bc-d1082433a7ed""}, statsOnLoad -> false)",null,null,null,1010-051943-vkcpjabi-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/dlt:18.3.7-delta-pipelines-aarch64-photon-dlt-release-dp-20260929-rc0-commit-e10efbb-image-5340b25


In [0]:
SELECT * FROM smart_claims_dev.01_bronze.claims